# Module 6: Model Comparison and Serialization

## Purpose
This notebook compares all trained classification models, selects the best-performing model, and prepares deployment-ready serialization artifacts without implementing any web or API components.

## Theory
Model comparison uses validation metrics such as accuracy, precision, recall, F1 score, and ROC AUC to identify the most reliable classifier for downstream use.

## Workflow
The workflow loads the trained models, evaluates each again on the test split, creates comparison tables and visualizations, selects the best model, saves the model and scaler, and verifies predictions.

## Interpretation
The comparison table and charts highlight which model generalizes best, while the saved artifacts are ready for Module 7 or further deployment workflows.

## Conclusion
The module produces a comparison leaderboard, visual evidence, and serialized artifacts for the final model and scaler.

In [ ]:
# Module 6: Model Comparison and Serialization
from pathlib import Path
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import pickle
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from utils.model_comparison import (
    build_comparison_table,
    compare_models,
    plot_accuracy,
    plot_precision,
    plot_recall,
    plot_f1,
    plot_auc,
    plot_training_time,
    plot_prediction_time,
    select_best_model,
    save_model,
    load_model,
    verify_model,
    run_comparison_pipeline,
)
from utils.model_training import load_training_data

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8")

# Load training data and compare all models
X_train, X_test, y_train, y_test = load_training_data()
results = compare_models(X_train, X_test, y_train, y_test)
comparison_table = build_comparison_table(results)
print(comparison_table)

# Select the best model
best_model_name, best_metrics = select_best_model(results)
print(f"Best model: {best_model_name}")

# Generate visual comparisons
plot_accuracy(results)
plot_precision(results)
plot_recall(results)
plot_f1(results)
plot_auc(results)
plot_training_time(results)
plot_prediction_time(results)

# Save final model and scaler
model = load_model(Path("../models/floods.save")) if Path("../models/floods.save").exists() else None
if model is None:
    from utils.model_training import decision_tree_model, random_forest_model, knn_model, xgboost_model
    if best_model_name == "XGBoost":
        model, _, _ = xgboost_model(X_train, X_test, y_train, y_test)
    elif best_model_name == "Decision Tree":
        model, _, _ = decision_tree_model(X_train, X_test, y_train, y_test)
    elif best_model_name == "Random Forest":
        model, _, _ = random_forest_model(X_train, X_test, y_train, y_test)
    else:
        model, _, _ = knn_model(X_train, X_test, y_train, y_test)

save_model(model)

# Verify prediction with loaded scaler
with Path("../models/transform.save").open("rb") as handle:
    scaler = pickle.load(handle)

sample_record = pd.DataFrame({
    "Temp": [29.0],
    "Humidity": [70.0],
    "Cloud Cover": [30.0],
    "ANNUAL": [3248.6],
    "Jan-Feb": [73.4],
    "Mar-May": [386.2],
    "Jun-Sep": [2122.8],
    "Oct-Dec": [666.1],
    "avgjune": [274.866667],
    "sub": [649.9],
})
scaled_input = pd.DataFrame(scaler.transform(sample_record), columns=sample_record.columns)
print(model.predict(scaled_input)[0])
